# Sample run for Fisher Market

In [1]:
using Pkg
Pkg.activate("../")

  Activating project at `~/workspace/ExchangeMarket.jl/scripts`


In [2]:
using Revise
using Random, SparseArrays, LinearAlgebra
using JuMP, MosekTools
using Plots, LaTeXStrings, Printf
import MathOptInterface as MOI

using ExchangeMarket

include("../tools.jl")
include("../plots.jl")
include("./setup.jl")
switch_to_pdf(; bool_use_html=false)

:pdf

In [12]:
Random.seed!(1)
n = 100
m = 1000
ρ = 1.0
# method_filter(name) = name ∈ [:LogBar]
# method_filter(name) = name ∈ [:Tât, :PropRes]
method_filter(name) = name ∈ [:PropRes]

table_time = []
results = []
results_phi = Dict()
results_ground = Dict()

f0 = FisherMarket(m, n; ρ=1.0, bool_unit=true, scale=30.0,
    sparsity=0.02,
    ε_br_play=1e-1
)
linconstr = LinearConstr(1, n, ones(1, n), [sum(f0.w)])

# -----------------------------------------------------------------------
# compute ground truth
# -----------------------------------------------------------------------
# μ₀ = 1e1
f1 = copy(f0)
p₀ = ones(n) * sum(f1.w) ./ (n)
# p₀ = ones(n) .* μ₀
x₀ = ones(n, m) ./ m
f1.x = copy(x₀)
f1.p .= p₀;

FisherMarket initialization started...
FisherMarket cost matrix initialized in 0.0001 seconds
FisherMarket initialized in 0.0003 seconds
FisherMarket initialization started...
FisherMarket cost matrix initialized in 0.0001 seconds
FisherMarket initialized in 0.0003 seconds


In [ ]:
# use proportional response method to compute ground truth
(name, method, kwargs) = method_kwargs[end]
alg = method(
    n, m, copy(p₀);
    linconstr=linconstr,
    kwargs...
)
traj = opt!(
    alg, f1;
    keep_traj=true,
    bool_init_phase=true,
    maxiter=2000,
    tol=1e-3
)
results_ground[ρ] = (alg, traj, f1)
results_phi[ρ] = copy(alg.p);
push!(results, ((name, ρ), (alg, traj[2:end], f1)))
push!(table_time, (n, m, name, ρ, traj[end].t))
validate(f1, alg)

# Primal-Dual IPM

In [11]:
include("./linear-pd-gpu.jl")
device = identity
fpd = copy(f0)
(; st, traj, kkt, status) = pd_ipm_gpu(
    fpd;
    μ₀=1.0, μ_scale=1.0,
    maxiter=50, tol=1e-7,
    device=device,
    # mode=:aug,
    mode=:neumann,
    # mode=:pcg,
    # mode=:schur,
)

pd_allocation_gpu!(fpd, st, kkt)
validate(fpd, st)

FisherMarket initialization started...
FisherMarket cost matrix initialized in 0.0001 seconds
FisherMarket initialized in 0.0003 seconds
-------------------------------------------------------------------------------------------------------------
                             Primal-Dual IPM (batched, device=CPU, n=100, m=1000)
-------------------------------------------------------------------------------------------------------------
 data transfer: 0.0104s | init: 0.0629s
-------------------------------------------------------------------------------------------------------------
    k |          μ |       |mc| |       |r₂| |       |r₃| |       |c₁| |       |c₂| |          α |    time(s)
-------------------------------------------------------------------------------------------------------------
    1 | +1.000e+00 |  1.021e+05 |  2.047e+04 |  0.000e+00 |  0.000e+00 |  1.110e-16 |     0.9980 |     0.5927
      |- t_res=0.0001  t_kkt=0.0003  t_build=0.0000  t_solve=0.0003  t_back=0.000

## Predictor-Correctors

In [13]:
include("./linear-pd-gpu.jl")
device = identity
fpd = copy(f0)
(; st, traj, kkt, status) = pd_ipm_gpu_pc(
    fpd;
    μ₀=1.0, μ_scale=1.0,
    maxiter=5, tol=1e-7,
    device=device,
    # mode=:aug,
    # mode=:neumann,
    # mode=:pcg,
    mode=:schur,
)

pd_allocation_gpu!(fpd, st, kkt)
validate(fpd, st)

FisherMarket initialization started...
FisherMarket cost matrix initialized in 0.0001 seconds
FisherMarket initialized in 0.0003 seconds
-------------------------------------------------------------------------------------------------------------
                             Primal-Dual IPM-PC (Mehrotra, device=CPU, n=100, m=1000)
-------------------------------------------------------------------------------------------------------------
 data transfer: 0.0139s | init: 0.0525s
-------------------------------------------------------------------------------------------------------------
    k |          μ |       |mc| |       |r₂| |       |r₃| |       |c₁| |       |c₂| |          α |    time(s)
-------------------------------------------------------------------------------------------------------------
    1 | +1.000e+00 |  1.021e+05 |  2.047e+04 |  0.000e+00 |  0.000e+00 |  1.110e-16 |     0.9986 |     0.4194
      |- t_res=0.0657  t_kkt=0.0835  t_pred=0.2025  t_corr=0.0005  σ=0.0000  

In [14]:
kkt.H_diag

100-element Vector{Float64}:
 1753.3383919550358
 1659.2521549695089
 1625.8104939884242
 1599.8049093041968
 1821.3068673211155
 1783.3910082238529
 1858.7320009165562
 1594.530698186757
 1795.8981786214028
 1613.2126719147898
    ⋮
 1653.3853536358813
 1695.9393066311054
 1995.150134184149
 1838.5511045935684
 1648.3457717832332
 1648.024205638837
 1704.4364119899863
 1418.7547275918823
 1666.6287010246554

In [24]:
sparse(kkt.Wc_scaled' * kkt.Wc_scaled)

1000×1000 SparseMatrixCSC{Float64, Int64} with 39112 stored entries:
⎡⣿⣿⣾⣿⣿⣿⣿⣿⣿⣿⣶⣷⣿⣿⣯⣿⣷⣿⣿⣿⣿⣿⣶⣿⣾⣿⣿⣿⣿⣿⣿⣿⣿⣿⣾⡿⣾⣿⣷⣗⎤
⎢⣾⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣷⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⢿⣿⢿⣿⣿⎥
⎢⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⢼⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣽⣿⣿⣿⣿⣷⎥
⎢⣯⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⣽⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣷⣿⣿⣿⣾⣷⎥
⎢⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⢿⣿⣿⣿⡿⣿⣿⣿⣿⣿⣿⣿⣿⡿⣿⎥
⎢⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⢿⣿⣿⣿⣿⣿⣿⣿⣿⣿⡟⎥
⎢⣼⣿⣽⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⢿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⣾⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣟⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣯⣿⣿⣿⣿⣿⡿⎥
⎢⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⡿⣿⣟⣿⣟⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⡿⣟⎥
⎢⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⡿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⣾⡿⣿⣿⣿⣟⣿⣿⣿⣿⣿⣿⣷⣿⣿⣿⣽⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⣾⣿⣿⣿⣿⣟⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣟⎥
⎣⢽⢿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⢿⣿⣿⣿⢾⣿⣿⣯⣿⠿⣿⣿⣿⡿⣿⣿⣿⣿⣿⢯⣿⣿⣿⣿⣿⢿⣿⣿⎦

In [20]:
eigvals(kkt.S_mat) ./ sum(eigvals(kkt.S_mat))

100-element Vector{Float64}:
 0.1327748311808183
 0.011224881012897389
 0.01080031861097738
 0.010416742137443596
 0.010412584788932006
 0.010298320594452329
 0.010038329116950746
 0.010032399496765555
 0.009978255061117134
 0.00995526605704716
 ⋮
 0.007507254159689949
 0.007500758784175348
 0.00749847277288944
 0.007435972171337184
 0.00730528624876943
 0.007272834451533337
 0.0071733105983492525
 0.0067288384506777104
 0.006606058199421806

# Other try

## ApproxLin

In [ ]:
# self-checking.
bool_run_approxlin = true
if bool_run_approxlin
    (name, method, kwargs) = [
        :LogBar,
        HessianBar,
        Dict(
            :tol => 1e-12, :maxiter => 20,
            :optimizer => ApproxLin,
            # :optimizer => CESConic,
            # :optimizer => DualLPConic,
            # :option_mu => :pred_corr,
            :option_mu => :normal,
            # :option_mu => :linear,
            :option_step => :affinesc,
            # :option_step => :homotopy,
            # :option_step => :logbar,
            # :linsys => :krylov,
            # :linsys => :DRq,
            :linsys => :direct,
        )
    ]
    f1 = copy(f0)
    p₀ = ones(n) * sum(f1.w) ./ (n)
    x₀ = ones(n, m) ./ m
    f1.x = copy(x₀)
    f1.p .= p₀
    alg = method(
        n, m, p₀;
        linconstr=linconstr,
        kwargs...
    )
    f1.ε_br_play .= 1e-1
    alg.μₗ = 1e-10
    traj = opt!(
        alg, f1;
        maxiter=200,
        pₛ=results_phi[ρ],
        keep_traj=true
    )
    push!(results, ((name, ρ), (alg, traj[2:end], f1)))
    push!(table_time, (n, m, name, ρ, traj[end].t))
end

## LSE (logsumexp-smoothed) dual solver

In [ ]:
min(10 / (log(n) + 6))

In [ ]:
# self-checking.
(name, method, kwargs) = [
    :LSE,
    HessianBar,
    Dict(
        :tol => 1e-12, :maxiter => 20,
        :optimizer => LSEResponse,
        # :option_mu => :normal,
        # :option_mu => :pred_corr,
        :option_mu => :nothing,
        # :option_step => :affinesc,
        :option_step => :logbar,
        # :option_step => :homotopy,
        :linsys => :direct,
        # :linsys => :krylov,
    )
]
f1 = copy(f0)
p₀ = ones(n) * sum(f1.w)
x₀ = ones(n, m) ./ m
f1.x = copy(x₀)
f1.p .= p₀
alg = method(
    n, m, p₀;
    linconstr=linconstr,
    kwargs...
)
f1.ε_br_play .= min(10 / (log(n) + 6), 6e-1)
alg.μₗ = 1e-20
traj = opt!(
    alg, f1;
    maxiter=20,
    pₛ=results_phi[ρ],
    keep_traj=true
)
push!(results, ((name, ρ), (alg, traj[2:end], f1)))
push!(table_time, (n, m, name, ρ, traj[end].t))

In [ ]:
(1.1 + √n) / (3.2 + √n)

In [ ]:
sum(alg.p)

In [ ]:
alg.y

## Validate the ground truth

In [ ]:
# for (ρ, (alg, traj, f1)) in results
for (ρ, (alg, traj, f1)) in results
    validate(f1, alg)
end

In [ ]:
[results[1][2][1].p results[2][2][1].p]

## Plot trajectory 

for each $\rho$, we plot the distance to ground truth $\|p - p^*\|$ of the trajectory.

In [ ]:
ffs = []

for attr in [:k]
    ρfmt = @sprintf("%+.2f", ρ)
    σfmt = @sprintf("%+.2f", ρ / (1 - ρ))
    fig = generate_empty(; shape=:wide)
    plot!(
        fig,
        ylabel=L"$\|\mathbf{p} - \mathbf{p}^*\|$",
        title=L"$\rho := %$ρfmt~(\sigma := %$σfmt)$",
        legendbackgroundcolor=RGBA(1.0, 1.0, 1.0, 0.8),
        yticks=10.0 .^ (-16:4:3),
        xtickfont=font(18),
        ytickfont=font(18),
        xscale=:identity,
        size=(1200, 600)
    )
    if attr == :k
        plot!(
            fig,
            xticks=[10, 20, 50, 100, 200, 500]
        )
    end
    for ((mm, _ρ), (alg, traj, f1)) in results
        if _ρ != ρ
            continue
        end
        traj_pp₊ = map(pp -> pp.D, traj)
        traj_tt₊ = map(pp -> getfield(pp, attr), traj)
        println(traj_pp₊)
        plot!(fig, traj_tt₊, traj_pp₊ .+ 1e-20, label=L"\texttt{%$mm}", linewidth=2, linestyle=:dash, markershape=:circle)
    end
    push!(ffs, fig)
end

In [ ]:
ff = plot(ffs...)

In [ ]:
savefig(ff, "/tmp/test_ipm_linear.pdf")